# Memory Systems in Agentic AI

## What you will build

A minimal but structurally correct memory architecture for an LLM agent using **LangGraph**:

- **Short-term memory**: the curated context passed to the model each step
- **Long-term semantic memory**: stable knowledge retrieved by embeddings
- **Long-term episodic memory**: experience/lessons retrieved and updated via reflection
- **State-aware gating**: memory selection changes by agent mode
- **Reflection write-back**: lessons are persisted and can influence future planning

## Requirements

- Set `OPENAI_API_KEY` in your environment.
- This notebook uses:
  - Embeddings: `text-embedding-3-small`
  - Chat model: `gpt-4o-mini`

## 1 — Install dependencies

We install:
- langgraph
- langchain-core
- langchain-community
- langchain-openai
- faiss-cpu
- tiktoken
- numpy

Skip if already installed.

In [1]:
%pip -q install langgraph langchain-core langchain-community langchain-openai faiss-cpu tiktoken numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


## 2 — Imports and OpenAI initialization

This cell:
1) Imports the core primitives (Document, FAISS, StateGraph).
2) Verifies `OPENAI_API_KEY` exists.
3) Initializes real OpenAI embeddings and a real chat model.

These are **real API calls** and will incur latency and cost.

In [2]:
from google.colab import userdata
import os
from typing import List, Dict, Any, TypedDict, Literal

from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langgraph.graph import StateGraph, END

os.environ["OPENAI_API_KEY"] = userdata.get("gpt_chat_key")

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Please set OPENAI_API_KEY before running this notebook.")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

## 3 — Define memory schema and seed semantic + episodic memory

We define `make_doc()` to attach metadata to each memory item.
Then we create two distinct long-term memory pools:

- Semantic memory: stable facts/principles
- Episodic memory: past events, outcomes, lessons

Keeping them separate reduces retrieval confusion and enables better policies.

In [3]:
def make_doc(text: str, mem_type: str, topic: str) -> Document:
    return Document(page_content=text, metadata={"mem_type": mem_type, "topic": topic})

semantic_docs = [
    make_doc("Blue-green deployment minimizes downtime by switching traffic between two identical environments.", "semantic", "deployment"),
    make_doc("Rolling updates gradually replace instances; misconfiguration can increase error rate during rollout.", "semantic", "deployment"),
    make_doc("Exponential backoff mitigates API rate limits by spacing retries.", "semantic", "reliability"),
    make_doc("Batching requests reduces rate-limit pressure and improves throughput.", "semantic", "reliability"),
    make_doc("A deployment checklist reduces execution drift and prevents missed steps.", "semantic", "process"),
]

episodic_docs = [
    make_doc("Episode: User rejected rolling update. Lesson: user prefers blue-green for minimal downtime.", "episodic", "deployment"),
    make_doc("Episode: Automation hit API rate limits. Lesson: add exponential backoff and batching.", "episodic", "reliability"),
    make_doc("Episode: Execution drift occurred. Lesson: enforce checklist and validate constraints before action.", "episodic", "process"),
]

(len(semantic_docs), len(episodic_docs))

(5, 3)

## 4 — Build long-term vector stores using real embeddings

This step embeds each document using OpenAI embeddings and builds two FAISS indexes locally:
- semantic_store
- episodic_store

We then create retrievers for each store.

In [4]:
semantic_store = FAISS.from_documents(semantic_docs, embeddings)
episodic_store = FAISS.from_documents(episodic_docs, embeddings)

semantic_retriever = semantic_store.as_retriever(search_kwargs={"k": 3})
episodic_retriever = episodic_store.as_retriever(search_kwargs={"k": 3})

## 5 — Define modes and agent state

We use 3 modes:
- planning
- execution
- reflection

AgentState includes:
- retrieved candidates (semantic_hits, episodic_hits)
- assembled short-term context (short_term_context)
- model output
- trace for observability

In [5]:
Mode = Literal["planning", "execution", "reflection"]

class AgentState(TypedDict, total=False):
    mode: Mode
    query: str
    semantic_hits: List[str]
    episodic_hits: List[str]
    short_term_context: Dict[str, Any]
    output: str
    trace: List[Dict[str, Any]]

## 6 — Retrieval policies (state-aware relevance gating)

Policy:
- planning: retrieve semantic + episodic
- execution: retrieve nothing (keeps costs down and avoids distraction)
- reflection: retrieve episodic

We also record trace events so the memory subsystem is debuggable.

In [6]:
def add_trace(state: AgentState, event: str, payload: Dict[str, Any]) -> AgentState:
    state.setdefault("trace", [])
    state["trace"].append({"event": event, "payload": payload})
    return state

def retrieve_semantic(state: AgentState) -> AgentState:
    mode = state["mode"]
    if mode == "planning":
        docs = semantic_retriever.invoke(state["query"])
        state["semantic_hits"] = [d.page_content for d in docs]
    else:
        state["semantic_hits"] = []
    return add_trace(state, "retrieve_semantic", {"mode": mode, "count": len(state["semantic_hits"])})

def retrieve_episodic(state: AgentState) -> AgentState:
    mode = state["mode"]
    if mode in ["planning", "reflection"]:
        docs = episodic_retriever.invoke(state["query"])
        state["episodic_hits"] = [d.page_content for d in docs]
    else:
        state["episodic_hits"] = []
    return add_trace(state, "retrieve_episodic", {"mode": mode, "count": len(state["episodic_hits"])})

## 7 — Context engineering (short-term memory assembly)

Retrieval gives candidates.
Context engineering decides what enters the model input.

We:
- cap to 2 items per memory type
- suppress injection in execution mode
- assemble a single short_term_context structure

This is the implementation of controlled influence.

In [7]:
def assemble_context(state: AgentState, cap: int = 2) -> AgentState:
    semantic = state.get("semantic_hits", [])[:cap]
    episodic = state.get("episodic_hits", [])[:cap]

    if state["mode"] == "execution":
        semantic, episodic = [], []

    state["short_term_context"] = {
        "mode": state["mode"],
        "query": state["query"],
        "semantic": semantic,
        "episodic": episodic,
    }
    return add_trace(state, "assemble_context", {
        "mode": state["mode"],
        "semantic_used": len(semantic),
        "episodic_used": len(episodic),
    })

## 8 — Generation

This node calls the chat model.
The model sees only short_term_context and produces mode-appropriate output.

- planning: plan with rationale
- execution: operational steps
- reflection: compact lesson summary suitable for storage

In [8]:
def generate(state: AgentState) -> AgentState:
    ctx = state["short_term_context"]

    prompt = (
        f"You are an agent operating in mode: {ctx['mode']}\n\n"
        f"User request or situation:\n{ctx['query']}\n\n"
        f"Selected semantic memory (facts/principles):\n{ctx['semantic']}\n\n"
        f"Selected episodic memory (past lessons/preferences):\n{ctx['episodic']}\n\n"
        "Instructions:\n"
        "- If mode=planning: produce a concise plan with constraints and rationale.\n"
        "- If mode=execution: produce step-by-step actions; keep it operational.\n"
        "- If mode=reflection: produce a compact lesson summary suitable to store as episodic memory.\n"
        "Output only the response for this mode."
    )

    response = llm.invoke(prompt)
    state["output"] = response.content
    return add_trace(state, "generate", {"chars": len(state["output"])})

## 9 — Reflection write-back (persist lessons to episodic memory)

If mode is reflection, we store the reflection output as a new episodic memory item.
This is the core "learning without retraining" loop.

In [9]:
def write_back(state: AgentState) -> AgentState:
    if state["mode"] == "reflection":
        episodic_store.add_documents([
            make_doc(state["output"], "episodic", "reflection_update")
        ])
        return add_trace(state, "write_back", {"added": True})
    return add_trace(state, "write_back", {"added": False})

## 10 — Build the LangGraph pipeline

We connect nodes:
semantic → episodic → assemble → generate → write_back → END

LangGraph makes memory selection steps explicit, inspectable, and easy to extend.

In [10]:
graph = StateGraph(AgentState)

graph.add_node("semantic", retrieve_semantic)
graph.add_node("episodic", retrieve_episodic)
graph.add_node("assemble", assemble_context)
graph.add_node("generate", generate)
graph.add_node("write_back", write_back)

graph.set_entry_point("semantic")
graph.add_edge("semantic", "episodic")
graph.add_edge("episodic", "assemble")
graph.add_edge("assemble", "generate")
graph.add_edge("generate", "write_back")
graph.add_edge("write_back", END)

app = graph.compile()

## 11 — Run lifecycle and inspect what was injected

We run:
1) planning
2) execution
3) reflection (writes back)
4) planning again (should be influenced by the new lesson)

We print:
- selected semantic and episodic items (short-term memory)
- the model output
- the internal trace

In [11]:
def run(mode: Mode, query: str) -> AgentState:
    state: AgentState = {"mode": mode, "query": query, "trace": []}
    return app.invoke(state)

def show(state: AgentState, label: str):
    print("\n" + "="*90)
    print(label)
    print("-"*90)
    ctx = state.get("short_term_context", {})
    print("MODE:", ctx.get("mode"))
    print("QUERY:", ctx.get("query"))
    print("\nSELECTED SEMANTIC:")
    for x in ctx.get("semantic", []):
        print("  -", x)
    print("\nSELECTED EPISODIC:")
    for x in ctx.get("episodic", []):
        print("  -", x)
    print("\nOUTPUT:\n", state.get("output", ""))
    print("\nTRACE:")
    for t in state.get("trace", []):
        print(" ", t["event"], t["payload"])

s1 = run("planning", "Deploy service with minimal downtime. Suggest a robust strategy.")
show(s1, "STEP 1: PLANNING")

s2 = run("execution", "Execute the deployment now. Provide the next operational steps.")
show(s2, "STEP 2: EXECUTION")

s3 = run("reflection", "Deployment failed due to API rate limits during cutover. Summarize lessons.")
show(s3, "STEP 3: REFLECTION (WRITE-BACK)")

s4 = run("planning", "Plan the next deployment avoiding the last failure and minimizing downtime.")
show(s4, "STEP 4: PLANNING (AFTER REFLECTION)")


STEP 1: PLANNING
------------------------------------------------------------------------------------------
MODE: planning
QUERY: Deploy service with minimal downtime. Suggest a robust strategy.

SELECTED SEMANTIC:
  - Blue-green deployment minimizes downtime by switching traffic between two identical environments.
  - A deployment checklist reduces execution drift and prevents missed steps.

SELECTED EPISODIC:
  - Episode: User rejected rolling update. Lesson: user prefers blue-green for minimal downtime.
  - Episode: Automation hit API rate limits. Lesson: add exponential backoff and batching.

OUTPUT:
 **Plan for Blue-Green Deployment with Minimal Downtime**

**Objective:** Deploy the new service version with minimal downtime using a blue-green deployment strategy.

**Constraints:**
1. **Identical Environments:** Ensure both blue and green environments are identical in configuration and resources.
2. **Traffic Switching:** Implement a seamless traffic switch to minimize user impact

## 12 — Verify the new episodic lesson is retrievable

After reflection write-back, we run an explicit episodic retrieval query.

This answers a practical debugging question:
- Did the reflection lesson actually enter long-term memory in a retrievable form?

In [12]:
docs = episodic_retriever.invoke("rate limits during cutover lesson summary")

print("Retrieved episodic items:")
for d in docs:
    print("-", d.page_content[:240].replace("\n", " "), "...")

Retrieved episodic items:
- Episode: Automation hit API rate limits. Lesson: add exponential backoff and batching. ...
- **Lesson Summary: Deployment Failures Due to API Rate Limits**  1. **Implement Exponential Backoff**: When encountering API rate limits, incorporate an exponential backoff strategy to manage retries effectively. This approach helps to reduc ...
- Episode: Execution drift occurred. Lesson: enforce checklist and validate constraints before action. ...
